
## GET Started With Faiss VECTOR STORE

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    DirectoryLoader
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv()

True

### DATA INGESTION AND PROCESSING

In [3]:
from langchain_core.documents import Document

sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is a branch of computer science that focuses on building systems
        capable of performing tasks that normally require human intelligence. These tasks include reasoning,
        learning, problem-solving, perception, and decision-making. AI is used in chatbots, recommendation systems,
        autonomous vehicles, healthcare diagnostics, and many other applications.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),

    Document(
        page_content="""
        Machine Learning (ML) is a subset of Artificial Intelligence that enables computers to learn
        patterns from data without being explicitly programmed. Common types of machine learning include
        supervised learning, unsupervised learning, and reinforcement learning. ML is widely used in spam filtering,
        fraud detection, price prediction, and recommendation systems.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),

    Document(
        page_content="""
        Natural Language Processing (NLP) is a field of Artificial Intelligence that enables computers
        to understand, interpret, and generate human language. NLP techniques are used in chatbots,
        language translation, sentiment analysis, text summarization, and question-answering systems.
        Modern NLP models are often based on transformer architectures such as BERT and GPT.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    ),

    Document(
        page_content="""
        Deep Learning (DL) is a specialized area of Machine Learning that uses neural networks with
        multiple hidden layers to learn complex patterns from large amounts of data. Deep Learning has
        achieved significant success in image recognition, speech recognition, medical imaging, and
        autonomous driving. Popular frameworks include TensorFlow and PyTorch.
        """,
        metadata={"source": "DL Basics", "page": 1, "topic": "DL"}
    )
]

print(f"Total Documents: {len(sample_documents)}")
print(f"MetaData:{sample_documents[0].metadata}")
print(f"PageContent:{sample_documents[0].page_content[:120]}")

Total Documents: 4
MetaData:{'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
PageContent:
        Artificial Intelligence (AI) is a branch of computer science that focuses on building systems
        capable o


### TEXT SPLITTING

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators=[" "],
)

chunks = text_splitter.split_documents(sample_documents)
chunks[0]

Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is a branch of computer science that focuses on building systems\n        capable of performing tasks that normally require human intelligence. These tasks include reasoning,\n        learning, problem-solving, perception, and decision-making. AI is used in chatbots, recommendation systems,\n        autonomous vehicles, healthcare diagnostics, and many other applications.')

### Initialize Embedding Model

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-miniLM-L6-V2"
)
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-miniLM-L6-V2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### Create FAISS VectoreStore

In [6]:
vectorestore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
vectorestore

In [7]:
vectorestore.save_local('faiss_index')
print("Vectorestore Save into 'FAISS_index' Directory")

Vectorestore Save into 'FAISS_index' Directory


In [8]:
loaded_vectorestor  = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True    
)

In [9]:
## Similarity Search
query = "What is deep learning "
results = vectorestore.similarity_search(query,k=3)
print(results)

[Document(id='42a6646e-5813-4783-a396-8828a3c50692', metadata={'source': 'DL Basics', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning (DL) is a specialized area of Machine Learning that uses neural networks with\n        multiple hidden layers to learn complex patterns from large amounts of data. Deep Learning has\n        achieved significant success in image recognition, speech recognition, medical imaging, and\n        autonomous driving. Popular frameworks include TensorFlow and PyTorch.'), Document(id='aabf4526-0d5e-415e-b4c8-077ac90cac63', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning (ML) is a subset of Artificial Intelligence that enables computers to learn\n        patterns from data without being explicitly programmed. Common types of machine learning include\n        supervised learning, unsupervised learning, and reinforcement learning. ML is widely used in spam filtering,\n        fraud detection, price prediction, and re

### Building Rag Chain With LECL

In [10]:
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm  = init_chat_model(
    model="groq:openai/gpt-oss-120b"
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000227BD1441A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000227BD1456A0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [11]:
simple_prompt = ChatPromptTemplate.from_template(
    """ 
    Answer The Following question Based only on the following 
    context:{context}
    Question:{question}
  Answer:  """)

In [12]:
retriever= vectorestore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":3}
)
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000227B2025400>, search_kwargs={'k': 3})

In [13]:
from typing import List
def format_docs(docs:List[Document]) -> str:
    """Format Document For Instruction into prompt """
    
    formatted = []
    for i,doc in enumerate(docs):
        source = doc.metadata.get('Source',"unknown")
        formatted.append(f"Document {i+1} (Source:{source}):\n{doc.page_content}")
        
    
    return "\n\n .join(formatted)"    
        

In [14]:
simple_rag_chain = (
    {"context":retriever | format_docs , "question":RunnablePassthrough()}
    | simple_prompt
    | llm
    | StrOutputParser()
)

In [15]:
conversational_prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful AI assistant.use the provided context to answer the question."),
        ("placeholder","{chat_history}"),
        ("human","Context: {context}\n\n Question : {input}"),
    ]
)

In [16]:
def create_conversational_rag():
    return(
        RunnablePassthrough.assign(
            context = lambda x : format_docs(retriever.invoke(x["input"]))
        )
        |conversational_prompt
        | llm
        | StrOutputParser()
    )
    
conversational_rag = create_conversational_rag()

In [17]:
## conversational rag
chat_history = []
q1 = "ehat do you mean by data structure"
a1 = conversational_rag.invoke({
    "input":q1,
    "chat_history":chat_history
})

print(a1)

**Data structure** is a way of organizing, storing, and accessing data in a computer program so that you can work with it efficiently.  
Think of it as a *container* that defines how elements are related to each other and what operations you can perform on them (e.g., add, remove, search, iterate).

---

### Common kinds of data structures

| Category | Example | Typical Use‑Case |
|----------|---------|------------------|
| **Linear** | **Array / List** – ordered collection of items accessed by index. <br>**Linked List** – nodes linked together, good for frequent insertions/removals. | Storing a sequence of values, iterating in order. |
| **Stack** | LIFO (last‑in‑first‑out) structure; push/pop operations. | Undo/redo, expression evaluation, recursion simulation. |
| **Queue** | FIFO (first‑in‑first‑out) structure; enqueue/dequeue. | Task scheduling, breadth‑first search, buffering. |
| **Tree** | Binary tree, AVL tree, B‑tree, Trie, etc. | Hierarchical data (file systems, DOM), fast 

In [18]:
from langchain.messages import AIMessage , HumanMessage
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [19]:
## conversational rag
chat_history = []
q1 = "who invented data structure"
a1 = conversational_rag.invoke({
    "input":q1,
    "chat_history":chat_history
})

print(a1)

The idea of “data structures” as a formal discipline didn’t spring from a single person, but the very first modern data structure that we still study today—the **linked list**—was invented in the mid‑1950s by a small team of pioneering computer scientists:

| Inventor(s) | Year | Contribution |
|-------------|------|--------------|
| **Allen Newell**, **Cliff Shaw**, and **Herbert A. Simon** | **1955** | Designed the linked list while building the **Logic Theory Machine** (the first program that could prove theorems). This structure allowed the program to store an arbitrarily long chain of symbols that could be easily extended or shortened, a hallmark of what we now call a “dynamic” data structure. |

### Why the linked list matters
- It was the first **dynamic, pointer‑based** structure, showing that a program could manage memory flexibly rather than being limited to fixed‑size arrays.
- It laid the groundwork for many later structures (stacks, queues, trees, graphs) that all rely on 